In [25]:
import pandas as pd

# 1) Paths
combined_path = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/targeton_primer_combined.csv"
oligo_path    = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/dna_oligo_raw(PrimersSeqOnly).csv"
output_path   = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/dna_oligo_with_meta.csv"

# 2) Load
combined = pd.read_csv(combined_path)
oligo    = pd.read_csv(oligo_path)

# 3) Determine join key
#    Try 'id' first, then '_pkey'. Adjust if your columns differ.
for key in ('id', '_pkey'):
    if key in combined.columns and key in oligo.columns:
        join_key = key
        break
else:
    raise KeyError("Neither 'id' nor '_pkey' found in both files.")

print(f"Using join key: {join_key!r}")

# 4) Which metadata fields to bring across?
meta_fields = [
    'genome_location',
    'product_size_bp',
    'primer_score',
    'qa_status',
    'tm_c',
    'gc_content',
    'computed_tm_c',
    'computed_gc_content'
]

# 5) Check overlap
common_keys = set(combined[join_key]).intersection(oligo[join_key])
print(f"Found {len(common_keys)} / {oligo.shape[0]} rows in dna_oligo_raw matching {combined_path}")

# 6) Subset the combined DF to just key + meta fields
combined_meta = combined[[join_key] + meta_fields].drop_duplicates(subset=join_key)

# 7) Merge into oligo
merged = oligo.merge(
    combined_meta,
    on=join_key,
    how='left',
    validate='many_to_one'  # each oligo row should match at most one meta row
)

# 8) Write out
merged.to_csv(output_path, index=False)
print("Merged file written to:", output_path)


Using join key: 'id'
Found 9482 / 13151 rows in dna_oligo_raw matching /Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/targeton_primer_combined.csv
Merged file written to: /Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/dna_oligo_with_meta.csv


In [27]:
import pandas as pd

# 1) Load your file
in_path  = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/dna_oligo_raw_meta.csv"
out_path = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/dna_oligo_cleaned.csv"

df = pd.read_csv(in_path)

# 2) Create Targeton from first 4 letters of name
df['Targeton'] = df['name'].astype(str).str[:4]

# 3) Rename columns
df = df.rename(columns={
    'name': 'primer_type',
    'bases': 'primer_bases'
})

# 4) Select and reorder columns
cols_to_keep = [
    'Targeton',
    'id',
    '_pkey',
    '_sync_key',
    'source_id',
    'acl_resource_id',
    'primer_type',
    'primer_bases',
    'genome_location',
    'product_size_bp',
    'primer_score',
    'qa_status',
    'tm_c',
    'gc_content',
    'computed_tm_c',
    'computed_gc_content'
]

# Filter to only those that exist in your DF
cols = [c for c in cols_to_keep if c in df.columns]

df_clean = df[cols]

# 5) Save out
df_clean.to_csv(out_path, index=False)
print("Done! Written cleaned file to:", out_path)





Done! Written cleaned file to: /Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/dna_oligo_cleaned.csv


In [30]:
import pandas as pd

# 1) load your already‐wide primers table
primers = pd.read_csv(
    "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/dna_oligo_cleaned.csv"
)

# 2) load the “all sequences” raw file
allseq = pd.read_csv(
    "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/dna_sequence_raw(AllSequences).csv"
)

# 3) drop any “pmin…” or TR<number> names
name_ser = allseq['name'].astype(str)
mask_exclude = (
       name_ser.str.contains('pmin', case=False, na=False)
    | name_ser.str.match(r'(?i)^TR\d+')
)
allseq = allseq[~mask_exclude].copy()

# 4) drop missing‐name & missing‐sequence rows
seq_col = 'sequence' if 'sequence' in allseq.columns else 'bases'
allseq = allseq[allseq['name'].notna() & allseq[seq_col].notna()]

# 5) extract Targeton = first 4 chars
allseq['Targeton'] = allseq['name'].str[:4]

# 6) normalize to underscores
allseq['name_u'] = allseq['name'].str.replace(r'\s+', '_', regex=True)

# 7) vectorized classification:
#    - anything with “grna” → gRNA
#    - anything with “target_region” → target_region
#    - anything with NO underscore at all → also target_region
pt = pd.Series(index=allseq.index, dtype="object")

# 7a) gRNA
pt.loc[ allseq['name_u'].str.contains('(?i)grna',    na=False) ] = 'gRNA'

# 7b) explicit target_region
pt.loc[ allseq['name_u'].str.contains('target_region', na=False) ] = 'target_region'

# 7c) pure 4-letter (no underscore) → target_region
pt.loc[ ~allseq['name_u'].str.contains('_', na=False) ] = 'target_region'

allseq['PrimerType'] = pt
allseq = allseq[allseq['PrimerType'].notna()]

# 8) drop any exact duplicates before pivoting
allseq = allseq.drop_duplicates(subset=['Targeton','PrimerType',seq_col])

# 9) pivot so each PrimerType (gRNA, target_region) is its own column
seq_wide = (
    allseq
    .pivot_table(
        index='Targeton',
        columns='PrimerType',
        values=seq_col,
        aggfunc='first'
    )
    .reset_index()
)
seq_wide.columns.name = None

# 10) merge on Targeton and move it to front
combined = primers.merge(seq_wide, on='Targeton', how='left')
cols = ['Targeton'] + [c for c in combined.columns if c!='Targeton']
combined = combined[cols]

# 11) save the final table
out = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/primers_plus_allseq_final.csv"
combined.to_csv(out, index=False)
print("Wrote:", out)


Wrote: /Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/primers_plus_allseq_final.csv


In [31]:
import pandas as pd

# 1. Paths to your files
primers_fp = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/primers_plus_allseq_final.csv"
guides_fp  = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/guide_rna_combined.csv"

# 2. Load
primers = pd.read_csv(primers_fp)
guides  = pd.read_csv(guides_fp)

# 3. Extract the four-letter Targeton from the guide names
#    (or use guides['targeton'] if that column already exists and is correct)
guides['Targeton'] = guides['name$'].astype(str).str[:4]

# 4. Pull out only the off_target_summary_data per Targeton
otr = (
    guides
    .loc[:, ['Targeton','off_target_summary_data']]
    .drop_duplicates(subset='Targeton')
)

# 5. Merge it onto your primers table
merged = primers.merge(otr, on='Targeton', how='left')

# 6. Move the new column to right after the gRNA column
cols = merged.columns.tolist()
gidx = cols.index('gRNA')
# pop out the new column, then re-insert
otr_col = cols.pop(cols.index('off_target_summary_data'))
cols.insert(gidx+1, otr_col)
merged = merged[cols]

# 7. Save
out_fp = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/primers_plus_allseq_with_offtargets.csv"
merged.to_csv(out_fp, index=False)

print("Done! Your merged table is at:", out_fp)


Done! Your merged table is at: /Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/primers_plus_allseq_with_offtargets.csv
